In [1]:
# Install PySpark into the active notebook kernel
#%pip install pyspark findspark
#%pip install pandas numpy scikit-learn xgboost matplotlib seaborn
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, exp, log1p, rand
import pandas as pd
import numpy as np
import os
import glob
#%pip install kagglehub
import kagglehub

In [2]:
# 1. Download latest version of the Kaggle dataset
path = kagglehub.dataset_download("rabieelkharoua/predict-conversion-in-digital-marketing-dataset")
print("Path to dataset files:", path)

# 2. Find and read the CSV file
csv_files = glob.glob(os.path.join(path, "*.csv"))
df_raw = pd.read_csv(csv_files[0])
print(f"Dataset loaded with shape: {df_raw.shape}")

# Inspect column names
print("Columns:", list(df_raw.columns))
df_raw.head()

Path to dataset files: /home/sagemaker-user/.cache/kagglehub/datasets/rabieelkharoua/predict-conversion-in-digital-marketing-dataset/versions/1
Dataset loaded with shape: (8000, 20)
Columns: ['CustomerID', 'Age', 'Gender', 'Income', 'CampaignChannel', 'CampaignType', 'AdSpend', 'ClickThroughRate', 'ConversionRate', 'WebsiteVisits', 'PagesPerVisit', 'TimeOnSite', 'SocialShares', 'EmailOpens', 'EmailClicks', 'PreviousPurchases', 'LoyaltyPoints', 'AdvertisingPlatform', 'AdvertisingTool', 'Conversion']


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


In [3]:
# Create working copy
df = df_raw.copy()

# Standardize column names (lowercase)
df.columns = [col.lower() for col in df.columns]

# 1. Define Treatment Variable (e.g., Targeted via Paid Social / Email vs Other)
# If 'campaignchannel' or similar column exists, create binary treatment
if 'campaignchannel' in df.columns:
    df['treatment_targeted'] = df['campaignchannel'].apply(lambda x: 1 if x in ['Social Media', 'Email'] else 0)
else:
    # Fallback to high-engagement exposure if multi-channel isn't explicit
    df['treatment_targeted'] = (df['clickthroughrate'] > df['clickthroughrate'].median()).astype(int)

# 2. Feature Transformations
df['log_income'] = np.log1p(df['income']) if 'income' in df.columns else 0
df['avg_time_per_visit'] = np.where(
    df['websitevisits'] > 0, 
    df['timeonsite'] / df['websitevisits'], 
    0
)

# 2. Re-scaled Engagement Index: (Visits * Avg Time per Visit)
# This gives a normalized measure of engagement depth per interaction
df['engagement_score'] = df['websitevisits'] * df['avg_time_per_visit']

# Display the newly engineered features
df[['customerid', 'websitevisits', 'timeonsite', 'avg_time_per_visit', 'engagement_score']].head()
# Save processed version to disk
os.makedirs("data", exist_ok=True)
df.to_csv("data/processed_data.csv", index=False)
print("Phase 2 Complete: Real Kaggle dataset processed and saved to data/processed_data.csv")

Phase 2 Complete: Real Kaggle dataset processed and saved to data/processed_data.csv


In [5]:
# Slightly increase variance / unobserved noise to create realistic overlap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


df = pd.read_csv("data/processed_data.csv")

# Identify feature columns (exclude target, IDs, and treatment)
exclude_cols = ['conversion', 'treatment_targeted', 'customerid', 'campaignchannel']
feature_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype in ['int64', 'float64']]

# Load processed data
X = df[feature_cols]
y_treatment = df['treatment_targeted']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y_treatment, test_size=0.2, random_state=42)



logit = (
    -1.2 
    + 0.000015 * df['income'] 
    + 0.12 * df['websitevisits'] 
    + 0.05 * df['emailopens'] 
    + np.random.normal(0, 1.8, size=len(df)) # Increased unobserved noise
)
prob_treatment = 1 / (1 + np.exp(-logit))

df['treatment_targeted'] = np.random.binomial(n=1, p=prob_treatment)

# Fit model
X = df[['age', 'income', 'websitevisits', 'timeonsite', 'avg_time_per_visit', 'engagement_score']]
y_treatment = df['treatment_targeted']

X_train, X_test, y_train, y_test = train_test_split(X, y_treatment, test_size=0.2, random_state=42)
model = xgb.XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=60, random_state=42)
model.fit(X_train, y_train)

test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Adjusted Propensity Model ROC-AUC (Ideal Overlap): {test_auc:.4f}")

# Update scored dataframe
df['propensity_score'] = model.predict_proba(X)[:, 1]
df.to_csv("data/scored_prospects.csv", index=False)

Adjusted Propensity Model ROC-AUC (Ideal Overlap): 0.7975


In [6]:
import os

# 1. Feature Engineering Module: src/data/features.py
os.makedirs("src/data", exist_ok=True)
with open("src/data/features.py", "w") as f:
    f.write('''import pandas as pd
import numpy as np

def engineer_features(df):
    """Engineers digital intent metrics and normalizes income."""
    df_out = df.copy()
    df_out.columns = [col.lower() for col in df_out.columns]
    
    if 'websitevisits' in df_out.columns and 'timeonsite' in df_out.columns:
        df_out['avg_time_per_visit'] = np.where(
            df_out['websitevisits'] > 0, 
            df_out['timeonsite'] / df_out['websitevisits'], 
            0
        )
        df_out['engagement_score'] = df_out['websitevisits'] * df_out['avg_time_per_visit']
    
    if 'income' in df_out.columns:
        df_out['log_income'] = np.log1p(df_out['income'])
        
    return df_out
''')

# 2. Propensity Model Module: src/models/propensity.py
os.makedirs("src/models", exist_ok=True)
with open("src/models/propensity.py", "w") as f:
    f.write('''import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def train_and_score(df, feature_cols, target_col):
    """Trains XGBoost propensity model and assigns scores and deciles."""
    X = df[feature_cols]
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = xgb.XGBClassifier(
        max_depth=3,
        learning_rate=0.03,
        n_estimators=60,
        random_state=42,
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)
    
    test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    
    df_scored = df.copy()
    df_scored['propensity_score'] = model.predict_proba(X)[:, 1]
    
    return model, df_scored, test_auc
''')

# 3. Causal Evaluation Module: src/evaluation/psm.py
os.makedirs("src/evaluation", exist_ok=True)
with open("src/evaluation/psm.py", "w") as f:
    f.write('''import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

def evaluate_psm_lift(df, treatment_col='treatment_targeted', outcome_col='conversion'):
    """Performs 1:1 Nearest-Neighbor PSM and calculates Average Treatment Effect (ATE)."""
    treated = df[df[treatment_col] == 1].copy()
    control = df[df[treatment_col] == 0].copy()
    
    nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
    nn.fit(control[['propensity_score']])
    
    distances, indices = nn.kneighbors(treated[['propensity_score']])
    matched_control = control.iloc[indices.flatten()].copy()
    
    treated_rate = treated[outcome_col].mean()
    control_rate = matched_control[outcome_col].mean()
    incremental_lift = treated_rate - control_rate
    
    return {
        'treated_conversion': treated_rate,
        'matched_control_conversion': control_rate,
        'incremental_lift_ate': incremental_lift
    }
''')

print("All production source modules successfully generated inside src/")

All production source modules successfully generated inside src/


In [7]:
with open("entrypoint.py", "w") as f:
    f.write('''import pandas as pd
import yaml
import os
from src.data.features import engineer_features
from src.models.propensity import train_and_score
from src.evaluation.psm import evaluate_psm_lift

def main():
    print("=== Starting FinTech Digital Acquisition Propensity Pipeline ===")
    
    # 1. Load Data
    data_path = "data/processed_data.csv"
    if not os.path.exists(data_path):
        print(f"Error: {data_path} not found. Please ensure data is loaded.")
        return
        
    df_raw = pd.read_csv(data_path)
    
    # 2. Feature Engineering
    df_engineered = engineer_features(df_raw)
    
    # 3. Model Training & Propensity Scoring
    feature_cols = ['age', 'income', 'websitevisits', 'timeonsite', 'avg_time_per_visit', 'engagement_score']
    model, df_scored, test_auc = train_and_score(df_engineered, feature_cols, target_col='treatment_targeted')
    
    print(f"Propensity Model Training Complete. Test ROC-AUC: {test_auc:.4f}")
    
    # Save Scored Prospects
    df_scored.to_csv("data/scored_prospects.csv", index=False)
    
    # 4. PSM Causal Lift Evaluation
    lift_results = evaluate_psm_lift(df_scored)
    
    print("\\n=== Causal Lift & Unit Economics Summary ===")
    print(f"Treated Group Conversion Rate:         {lift_results['treated_conversion']:.2%}")
    print(f"Matched Control Group Conversion Rate: {lift_results['matched_control_conversion']:.2%}")
    print(f"Incremental Conversion Lift (ATE):      {lift_results['incremental_lift_ate']:.2%}")
    print("===============================================================")

if __name__ == "__main__":
    main()
''')

print("entrypoint.py successfully created!")

entrypoint.py successfully created!
